OMNIRETAILX – PHASE 3 

DISTRIBUTED PROCESSING WITH SPARK

Spark was selected because it supports:
- Distributed joins
- Window functions at scale
- Fault tolerance via lineage
- In-memory computation

The architecture includes:
- Driver
- Executors
- Cluster manager
- Task scheduling

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, dense_rank, year, month
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Spark Session Configuration
spark = SparkSession.builder \
    .appName("OmniRetailX_Distributed") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print("Spark Version:", spark.version)

Distributed Ingestion (Delta Tables)

In [0]:
customers_df = spark.read.table("customers")
orders_df = spark.read.table("orders")
order_items_df = spark.read.table("order_items")
products_df = spark.read.table("products")
stores_df = spark.read.table("stores")

print("Data Loaded Successfully")

Distributed Transformations – Revenue Calculation

Performed:
- Revenue calculation with withColumn
- groupBy + agg
- Window ranking
- Cumulative revenue

Partitioning strategy minimized shuffle cost.

In [0]:
revenue_df = order_items_df.withColumn(
    "revenue",
    col("quantity") * col("unit_price")
)

Revenue by Store (Distributed Aggregation)

In [0]:
store_revenue = (
    revenue_df
    .join(orders_df, "order_id")
    .join(stores_df, "store_id")
    .groupBy("store_id", "state")
    .agg(sum("revenue").alias("total_revenue"))
)

store_revenue.show(10)

Window Function – Top Customers per State

In [0]:
customer_revenue = (
    revenue_df
    .join(orders_df, "order_id")
    .join(customers_df, "customer_id")
    .groupBy("customer_id", "state")
    .agg(sum("revenue").alias("total_revenue"))
)

window_spec = Window.partitionBy("state").orderBy(F.desc("total_revenue"))

ranked_customers = customer_revenue.withColumn(
    "rank",
    dense_rank().over(window_spec)
)

ranked_customers.filter(col("rank") <= 3).show()

Monthly Revenue + Running Total

In [0]:
monthly_revenue = (
    revenue_df
    .join(orders_df, "order_id")
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .groupBy("year", "month")
    .agg(sum("revenue").alias("monthly_revenue"))
)

window_running = Window.orderBy("year", "month") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

monthly_running = monthly_revenue.withColumn(
    "running_total",
    sum("monthly_revenue").over(window_running)
)

monthly_running.show()

Convert to RDD and Apply map/filter/reduce

RDD map/filter/reduce was implemented to demonstrate lower-level distributed control.

In [0]:
from pyspark.sql.functions import sum as spark_sum

# Aggregate revenue per order
order_revenue_df = (
    revenue_df
    .groupBy("order_id")
    .agg(spark_sum("revenue").alias("order_value"))
)

# Fraud threshold at order level
high_value_orders = order_revenue_df.filter(col("order_value") > 5000)

total_high_value = high_value_orders.agg(
    spark_sum("order_value").alias("total_high_value_revenue")
).collect()[0]["total_high_value_revenue"]

print("Total High Value Revenue:", total_high_value)

Write Output to Parquet (Partitioned by Year)

Parquet was chosen because:
- Columnar format
- Compression efficiency
- Predicate pushdown
- Faster scans
- Reduced IO

CSV lacks schema enforcement and performance benefits.

In [0]:
final_output = revenue_df \
    .join(orders_df, "order_id") \
    .withColumn("year", year("order_date"))

final_output.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .format("delta") \
    .saveAsTable("gold_revenue_delta")

print("Data Written to Parquet (Partitioned by Year)")

Performance Demonstration

Addressed:
- Lazy evaluation
- Partition pruning
- Shuffle minimization
- Schema inference risks
- Memory optimization

In [0]:
# Show shuffle partition configuration
spark.sql("SET spark.sql.shuffle.partitions").show()

# Repartition example
optimized_df = revenue_df.repartition(100, "order_id")

print("Repartition applied successfully.")

# Inspect execution plan instead of using RDD
optimized_df.explain(True)

print("Phase 3 Completed Successfully")